# 02 — Approche semi-supervisée et comparaison avec une baseline supervisée

**Mission BrainScanAI — *CurelyticsIA*** &nbsp;·&nbsp; *Option B*

Ce second notebook met en œuvre l'**étape 4** de la mission :

1. Charger les pseudo-labels faibles (issus du clustering du notebook 01) et
   les 100 labels forts (sans jamais mélanger les deux jeux).
2. Réaliser un split *train / test* stratifié sur les labels forts (le test set
   restera **strictement inconnu** des deux modèles).
3. Entraîner :
   - une **baseline supervisée pure** (ResNet18 + tête 2 classes, fine-tuné
     uniquement sur les 80 images fortement labellisées du train) ;
   - une **approche semi-supervisée** (même architecture, pré-entraînée sur les
     ~1 400 pseudo-labels faibles, puis fine-tunée sur les mêmes 80 images).
4. Comparer les performances : accuracy, macro-F1, **recall de la classe
   *cancer*** (priorité métier), précision, ROC AUC, matrice de confusion.
5. Conclusion / *Definition of Done*.

## 0. Imports & configuration

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from curelyticsia.config import (  # noqa: E402
    CLASS_TO_INDEX,
    CLASSES,
    ClusteringConfig,
    FIGURES_DIR,
    FeatureConfig,
    PROCESSED_DIR,
    SEED,
    TrainingConfig,
    ensure_dirs,
    set_global_seeds,
)
from curelyticsia.features.extractor import load_cached_features  # noqa: E402
from curelyticsia.viz.plots import (  # noqa: E402
    plot_confusion_matrix,
    plot_history,
    plot_metrics_bar,
)

ensure_dirs()
set_global_seeds(SEED)
print("Classes :", CLASSES, "→", CLASS_TO_INDEX)

## 1. Chargement des features et des labels

In [ ]:
features, index_df = load_cached_features(FeatureConfig().cache_path)
weak = pd.read_csv(ClusteringConfig().weak_labels_path)
print(f"features            : {features.shape}")
print(f"index_df            : {index_df.shape}")
print(f"pseudo-labels faibles : {len(weak)}")

labeled_df = index_df[index_df["split"] == "labeled"].copy()
labeled_df["label_index"] = labeled_df["label_name"].map(CLASS_TO_INDEX).astype(int)
print("\nDistribution labels forts :")
print(labeled_df["label_name"].value_counts())

### 1.1 Garde-fous : aucun chevauchement entre faibles et forts

In [ ]:
strong_ids = set(labeled_df["image_id"])
weak_ids = set(weak["image_id"])
overlap = strong_ids & weak_ids
assert not overlap, f"Chevauchement détecté : {overlap}"
print("OK — pas de chevauchement entre labels forts et faibles")

## 2. Split train / test stratifié sur les labels forts

In [ ]:
train_idx, test_idx = train_test_split(
    labeled_df.index,
    test_size=TrainingConfig().test_size,
    stratify=labeled_df["label_index"],
    random_state=SEED,
)
strong_train = labeled_df.loc[train_idx]
strong_test = labeled_df.loc[test_idx]

print(f"strong_train : {len(strong_train)} ({strong_train['label_name'].value_counts().to_dict()})")
print(f"strong_test  : {len(strong_test)} ({strong_test['label_name'].value_counts().to_dict()})")
print(f"weak (faibles): {len(weak)} ({weak['weak_label_name'].value_counts().to_dict()})")

## 3. Stratégie A — Baseline supervisée pure

On entraîne un ResNet18 pré-entraîné ImageNet, dont on remplace la tête par
une couche linéaire 2 classes. Le réseau est ré-entraîné de bout en bout sur
les 80 images du train fortement labellisé.

In [ ]:
from curelyticsia.models.semi_supervised import train_supervised

cfg = TrainingConfig()
print(cfg)

sup_model, sup_report = train_supervised(
    train_paths=strong_train["path"].tolist(),
    train_labels=strong_train["label_index"].tolist(),
    test_paths=strong_test["path"].tolist(),
    test_labels=strong_test["label_index"].tolist(),
    class_names=list(CLASSES),
    cfg=cfg,
)
print(json.dumps(
    {k: v for k, v in sup_report.__dict__.items() if k not in ("history",)},
    indent=2, default=lambda o: float(o) if isinstance(o, np.floating) else o,
))

## 4. Stratégie B — Semi-supervisé (faible → fort)

Le réseau démarre avec les mêmes poids ImageNet, est **pré-entraîné** sur les
~1 400 pseudo-labels (apprentissage de structures visuelles communes via les
clusters identifiés à l'étape 3), puis **fine-tuné** sur les 80 images du
train fortement labellisé avec un *learning rate* divisé par deux.

In [ ]:
from curelyticsia.models.semi_supervised import train_semi_supervised

semi_model, semi_report = train_semi_supervised(
    weak_paths=weak["path"].tolist(),
    weak_labels=weak["weak_label_index"].astype(int).tolist(),
    strong_train_paths=strong_train["path"].tolist(),
    strong_train_labels=strong_train["label_index"].tolist(),
    strong_test_paths=strong_test["path"].tolist(),
    strong_test_labels=strong_test["label_index"].tolist(),
    class_names=list(CLASSES),
    cfg=cfg,
)
print(json.dumps(
    {k: v for k, v in semi_report.__dict__.items() if k not in ("history",)},
    indent=2, default=lambda o: float(o) if isinstance(o, np.floating) else o,
))

## 5. Comparaison des deux stratégies

In [ ]:
def to_run(report) -> dict[str, float]:
    return {
        "accuracy": report.accuracy,
        "f1_macro": report.f1_macro,
        "recall_cancer": report.recall_per_class.get("cancer", float("nan")),
        "precision_cancer": report.precision_per_class.get("cancer", float("nan")),
        "f1_cancer": report.f1_per_class.get("cancer", float("nan")),
        "roc_auc": report.roc_auc if report.roc_auc is not None else float("nan"),
    }

runs = {"Supervisé pur": to_run(sup_report), "Semi-supervisé": to_run(semi_report)}
comparison = pd.DataFrame(runs).T
comparison.style.format("{:.3f}")

In [ ]:
_ = plot_metrics_bar(
    runs,
    metric_name="f1_macro",
    title="Macro-F1 — Supervisé vs Semi-supervisé",
    save_path=FIGURES_DIR / "compare_f1.png",
)
_ = plot_metrics_bar(
    runs,
    metric_name="recall_cancer",
    title="Rappel sur la classe cancer (priorité métier)",
    save_path=FIGURES_DIR / "compare_recall_cancer.png",
)

### 5.1 Matrices de confusion

In [ ]:
_ = plot_confusion_matrix(
    sup_report.confusion_matrix,
    class_names=list(CLASSES),
    title="Confusion — Supervisé pur",
    save_path=FIGURES_DIR / "cm_supervised.png",
)
_ = plot_confusion_matrix(
    semi_report.confusion_matrix,
    class_names=list(CLASSES),
    title="Confusion — Semi-supervisé",
    save_path=FIGURES_DIR / "cm_semi.png",
)

### 5.2 Historique des phases d'entraînement

In [ ]:
def history_to_df(report, label: str) -> pd.DataFrame:
    rows = []
    for i, log in enumerate(report.history, start=1):
        rows.append({
            "step": i,
            "phase": f"{label}/{log.phase}",
            "train_loss": log.train_loss,
            "train_acc": log.train_acc,
        })
    return pd.DataFrame(rows)

hist_df = pd.concat([
    history_to_df(sup_report, "supervised"),
    history_to_df(semi_report, "semi-supervised"),
], ignore_index=True)

_ = plot_history(hist_df, save_path=FIGURES_DIR / "training_history.png")
hist_df.tail(10)

## 6. Discussion : justification de l'approche semi-supervisée

**Pourquoi la semi-supervision est-elle pertinente ici ?**

1. Le coût d'annotation médicale est **élevé** (radiologues experts, temps,
   responsabilité). Le budget *labellisation IA* est de **300 €** sur ce
   dataset et ne permettra pas d'annoter manuellement 1 500 IRM
   supplémentaires.
2. Les 1 400 images non annotées **portent de l'information** sur la structure
   visuelle des IRM (forme du crâne, position cérébrale, contraste type IRM).
   Le pré-entraînement sur les pseudo-labels stabilise la représentation et
   réduit le sur-apprentissage de la baseline 100 % supervisée sur seulement
   80 images d'entraînement.
3. La comparaison directe (mêmes données de test, même architecture, même
   *seed*) montre l'apport — ou ses limites — de l'approche.

**Hyperparamètres ajustés** :

- *learning rate* (1e-4 sur la phase fortement labellisée, divisé par 2 lors du
  fine-tuning pour préserver le pré-entraînement) ;
- nombre d'époques (5 en pré-entraînement faible, 8 en fine-tuning fort) ;
- *batch size* à 16 (compromis sur des images 224×224 en CPU).

**Erreur la plus coûteuse** : un **faux négatif sur cancer** (manquer une
tumeur) coûte plus cher qu'un faux positif (re-vérifier une IRM saine). Les
métriques mises en avant sont donc **`recall_cancer`** puis **`f1_cancer`**,
plutôt que l'accuracy globale qui peut masquer un déséquilibre de classes.

## 7. Bonus — Comparaison avec sklearn `LabelPropagation`

La consigne suggère également les méthodes basées sur les graphes
(*label propagation*). On s'en sert ici comme **second avis** sur les features
ResNet déjà extraites — sans avoir à réentraîner un CNN.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import LabelPropagation

# On reconstruit truth aligné sur l'ordre de features.
truth_full = np.full(len(features), -1, dtype=int)
for idx, row in labeled_df.iterrows():
    pos = int(np.where(index_df["image_id"].values == row["image_id"])[0][0])
    truth_full[pos] = int(row["label_index"])

# On ne fournit à LabelPropagation que les labels du **train fort** + les
# features ; le test reste donc inconnu — comparaison honnête.
train_ids = set(strong_train["image_id"].tolist())
test_ids = set(strong_test["image_id"].tolist())

semi_truth = np.where(
    [iid in train_ids for iid in index_df["image_id"]],
    truth_full,
    -1,
)

X = StandardScaler().fit_transform(features)
lp = LabelPropagation(kernel="knn", n_neighbors=10, max_iter=200)
lp.fit(X, semi_truth)

# Évaluation sur le test set fortement labellisé.
test_mask = np.array([iid in test_ids for iid in index_df["image_id"]])
y_true_test = truth_full[test_mask]
y_pred_test = lp.transduction_[test_mask]
from sklearn.metrics import accuracy_score, f1_score, recall_score
print("LabelPropagation — accuracy   :", accuracy_score(y_true_test, y_pred_test))
print("LabelPropagation — macro F1   :", f1_score(y_true_test, y_pred_test, average="macro"))
print("LabelPropagation — recall cancer :", recall_score(y_true_test, y_pred_test, pos_label=CLASS_TO_INDEX["cancer"]))

## 8. *Definition of Done*

In [ ]:
done_table = pd.DataFrame(
    [
        ("ARI clustering vs labels forts ≥ 0.10", "voir notebook 01"),
        ("F1 macro semi-sup ≥ F1 macro supervisé", f"{semi_report.f1_macro:.3f} vs {sup_report.f1_macro:.3f}"),
        ("Recall cancer ≥ 0.90", f"sup={sup_report.recall_per_class.get('cancer', float('nan')):.3f} ; semi={semi_report.recall_per_class.get('cancer', float('nan')):.3f}"),
        ("Tests pytest verts", "voir QA"),
        ("Notebooks ré-exécutables", "uv run jupyter nbconvert --execute"),
    ],
    columns=["Critère", "Valeur observée"],
)
done_table

In [ ]:
# Sauvegarde du rapport JSON consommé par le support de présentation.
out = {
    "supervised": to_run(sup_report),
    "semi_supervised": to_run(semi_report),
    "training_config": cfg.__dict__,
}
out_path = PROCESSED_DIR / "training_report.json"
out_path.write_text(json.dumps(out, indent=2, default=lambda o: float(o) if isinstance(o, np.floating) else o), encoding="utf-8")
print(f"Rapport sauvegardé : {out_path}")

**Conclusion**

- L'approche semi-supervisée s'appuie sur les pseudo-labels issus du
  clustering pour exploiter les 1 400 IRM non annotées sans coût d'annotation
  supplémentaire.
- La comparaison directe avec la baseline supervisée pure (mêmes données,
  même test) permet de quantifier le gain.
- Les recommandations pour le passage à l'échelle (4 M d'images, 5 000 €) sont
  détaillées dans le support de présentation : faisabilité, choix techniques
  (batch GPU, infrastructure, *active learning*), risques et conditions.